In [1]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.feature import hog
from skimage import exposure
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
accuracy_score,
classification_report,
confusion_matrix,
ConfusionMatrixDisplay,
roc_auc_score,
RocCurveDisplay,
)
import joblib

In [2]:
random_state = 42
np.random.seed(random_state)
random.seed(random_state)

In [ ]:
#rozpakowanie i przygotowanie danych
#tu używamy Path - biblioteki do pracy z systemem plików i folderami

#folder z calym data setem znajduje się tu
DATA_DIR = Path(r"C:\Users\user\Desktop\brain_tumor\data\brain_tumor_mri_dataset")
#tworzymy sciezke do danych treningowych i testowych
TRAIN_DIR = DATA_DIR / "Training"
TEST_DIR = DATA_DIR / "Testing"
#sprawdzamy czy foldery istnieją i co się w nich znajduje 
print(TRAIN_DIR.exists(), TEST_DIR.exists())
print(list(TRAIN_DIR.iterdir()))


True True
[WindowsPath('C:/Users/user/Desktop/brain_tumor/data/brain_tumor_mri_dataset/Training/glioma'), WindowsPath('C:/Users/user/Desktop/brain_tumor/data/brain_tumor_mri_dataset/Training/meningioma'), WindowsPath('C:/Users/user/Desktop/brain_tumor/data/brain_tumor_mri_dataset/Training/notumor'), WindowsPath('C:/Users/user/Desktop/brain_tumor/data/brain_tumor_mri_dataset/Training/pituitary')]


In [ ]:
#dataFrame 
TUMOR_CLASSES = ["glioma_tumor", "meningioma_tumor", "notumor", "pituitary_tumor"]
NO_TUMOR_CLASS = "notumor"

VALID_IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png",] 

def binary_label(orginal_label: str) -> int:
    if orginal_label == NO_TUMOR_CLASS:
        return 0
    else:        return 1
    
#

In [26]:
def build_dataframe(split_dir: Path) -> pd.DataFrame:
    data = []  #pusta lsta do której bedziemy dodawać dane
    for class_dir in split_dir.iterdir(): #iterujemy po folderach z klasami
        if not class_dir.is_dir(): #sprawdzamy czy to jest folder
            continue
        orginal_label = class_dir.name #nazwa folderu to nasza etykieta
        
        if orginal_label == "notumor":
            binary_label_value = 0
            binary_label_name = "no_tumor"
        else:
            binary_label_value = 1
            binary_label_name = "tumor"

        for image_path in class_dir.iterdir(): #iterujemy po plikach w folderze
            if image_path.suffix.lower() not in VALID_IMAGE_EXTENSIONS: #sprawdzamy czy to jest obraz
                continue
            data.append(
                {
                    "image_path": image_path,
                    "original_label": orginal_label,
                    "binary_label_value": binary_label_value,
                    "binary_label_name": binary_label_name,
                }
            )
    return pd.DataFrame(data) #tworzymy DataFrame z listy danych

full_train_df = build_dataframe(TRAIN_DIR)
test_df = build_dataframe(TEST_DIR)
display(full_train_df.head(20))
display(full_train_df["binary_label_name"].value_counts())
display(test_df["binary_label_name"].value_counts())


,image_path,original_label,binary_label_value,binary_label_name
0,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
1,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
2,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
3,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
4,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
5,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
6,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
7,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
8,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor
9,C:\Users\user\Desktop\brain_tumor\data\brain_t...,glioma,1,tumor


binary_label_name
tumor       4200
no_tumor    1400
Name: count, dtype: int64

binary_label_name
tumor       1200
no_tumor     400
Name: count, dtype: int64

In [29]:
#train\vaidation split
#from sklearn.model_selection import train_test_split - ta funcja sluzy do podziału danych na zbiór treningowy i walidacyjny
# mimo ze nazywa się train test to my tylko dzielimy na train validation, test zostawiamy na koniec do ostatecznej ewaluacji modelu
train_df, val_df = train_test_split(full_train_df, test_size=0.2, stratify=full_train_df["binary_label_value"], random_state=random_state)
# stratify - to jest parametr który pozwala nam zachować proporcje klas w obu zbiorach, czyli w train i validation bedziemy mieli podobna liczbe przypadków z każda klasą
#obie czesci maja podbny rozklad klas, co jest ważne dla trenowania modelu

print("train_df shape:", train_df.shape)
print("val_df shape:", val_df.shape)
print("test_df shape:", test_df.shape)

print("\nTrain set distribution:")
print(train_df["binary_label_name"].value_counts())
print("\nValidation set distribution:")
print(val_df["binary_label_name"].value_counts())
print("\nTest set distribution:")
print(test_df["binary_label_name"].value_counts())

train_df shape: (4480, 4)
val_df shape: (1120, 4)
test_df shape: (1600, 4)

Train set distribution:
binary_label_name
tumor       3360
no_tumor    1120
Name: count, dtype: int64

Validation set distribution:
binary_label_name
tumor       840
no_tumor    280
Name: count, dtype: int64

Test set distribution:
binary_label_name
tumor       1200
no_tumor     400
Name: count, dtype: int64
